In [1]:
from datasets import load_dataset
from data_structures import Example
from hierarchical_optimizer import HierarchicalOptimizer

SQUAD_METRICS = [
    {"name": "exact_match",         "weight": 0.8, "stage": 1},
    {"name": "token_f1",            "weight": 0.2, "stage": 1},
]

/Users/kateaver/idea1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/kateaver/idea1/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/kateav

In [2]:
def squad_v2_to_examples(data):
    examples = []
    for item in data:
        input_text = (f'# Question: \n {item["question"]} \n'
                      f'# Context: \n {item["context"]} \n')
        all_answers = item['answers']['text']
        expected_output = all_answers[0] if all_answers else 'No answer'
        examples.append(Example(
            input_text=input_text,
            expected_output=expected_output,
            metadata={"all_answers": all_answers},
        ))
    return examples

def get_squad_v2_data(train_num: int, val_num: int, test_num: int):
    ds_train = load_dataset('rajpurkar/squad_v2', split='train')
    ds_test = load_dataset('rajpurkar/squad_v2', split='validation')
    
    ds_train = ds_train.shuffle()
    ds_test = ds_test.shuffle()

    split = ds_train.train_test_split(test_size=0.1, seed=42)
    ds_train = split['train']
    ds_val = split['test']

    train_split = ds_train.select(range(train_num))
    val_split = ds_val.select(range(val_num))
    test_split = ds_test.select(range(test_num))

    train_examples = squad_v2_to_examples(train_split)
    validation_examples = squad_v2_to_examples(val_split)
    test_examples = squad_v2_to_examples(test_split)

    return train_examples, validation_examples, test_examples

def data_fabric(dataset: str = 'squad_v2', train_num: int = 150, val_num: int = 100, test_num: int = 300):
    squad_v2_initial_prompt = """Read the context carefully and answer the question by extracting the exact answer span from the context. Output only the minimal text from the context that answers the question. If the question cannot be answered based on the context, output exactly 'No answer'."""

    train_examples, validation_examples, test_examples = get_squad_v2_data(train_num, val_num, test_num)
    initial_prompt = squad_v2_initial_prompt
    
    return train_examples, validation_examples, test_examples, initial_prompt

In [3]:
train_examples, validation_examples, test_examples, initial_prompt = data_fabric()
print("Dataset prepared:")
print(f"  Train: {len(train_examples)} examples")
print(f"  Validation: {len(validation_examples)} examples")
print(f"  Test: {len(test_examples)} examples")

Dataset prepared:
  Train: 150 examples
  Validation: 100 examples
  Test: 300 examples


In [4]:
print("Initial prompt:")
print("-" * 60)
print(initial_prompt)
print("-" * 60)

Initial prompt:
------------------------------------------------------------
Read the context carefully and answer the question by extracting the exact answer span from the context. Output only the minimal text from the context that answers the question. If the question cannot be answered based on the context, output exactly 'No answer'.
------------------------------------------------------------


In [5]:
TASK_DESCRIPTION = (
    "Extractive question answering: given a context paragraph and a question, "
    "extract the exact answer span from the context. If the question is unanswerable based "
    "on the context, output exactly 'No answer'."
)

optimizer = HierarchicalOptimizer(
    metrics_config=SQUAD_METRICS,
    task_description=TASK_DESCRIPTION,
)

In [6]:
best_node = optimizer.optimize(
    initial_prompt=initial_prompt,
    train_examples=train_examples,
    validation_examples=validation_examples,
    test_examples=test_examples,
    save_dir="./optimization_results",
)

Evaluating initial prompt...
[diag] initial node: prompt_id=ab48b463fef6 len=262 chars
[diag] Prompt text: Read the context carefully and answer the question by extracting the exact answer span from the context. Output only the minimal text from the context that answers the question. If the question cannot be answered based on the context, output exactly 'No answer'.
[diag] evaluate_node: node_id=1a3861d6-bd1e-416d-a2fa-ec1c35c3c011 gen=0 source=initial split=validation stage=1 examples=100 seed_offset=0
[diag] evaluate_prompt: execute=True sample=True stage=1 incoming=100 will_use=100 weights={exact_match:0.8, token_f1:0.2} llm_calls_at_start=0
[diag] execute_prompt_batch: total=100 batch_size=25 n_batches=4 llm_calls_before=0
[diag]   batch 1/4: OK (25/100 done, llm_calls=1)
[diag]   batch 2/4: OK (50/100 done, llm_calls=2)
[diag]   batch 3/4: OK (75/100 done, llm_calls=3)
[diag]   batch 4/4: OK (100/100 done, llm_calls=4)
[diag] execute_prompt_batch done: llm_calls_delta=4 total_cal

In [7]:
report = optimizer.get_optimization_report()
print('Optimization generations summary:')
for entry in report['optimization_log']:
    print(f"  Generation {entry['generation']}: time {entry['time']:.2f}s, best_score {entry['best_score']:.3f}")

local_stats = report['component_statistics']['local_optimizer']
print('Local optimizer summary:')
print(f"  Total iterations recorded: {local_stats.get('total_iterations', 0)}")
avg_it = local_stats.get('avg_iteration_time')
if avg_it is not None:
    print(f"  Avg iteration time: {avg_it:.2f}s")
else:
    print('  Avg iteration time: N/A')
print(f"  Total LLM calls attributed to local iterations: {local_stats.get('total_llm_calls_by_local', 0)}")
print('Per-iteration breakdown:')
for s in local_stats.get('iteration_stats', []):
    print(f"  Iter {s['iteration']}: time {s['time']:.2f}s, llm_calls {s['llm_calls']}")

Optimization generations summary:
  Generation 1: time 568.32s, best_score 0.511
  Generation 2: time 613.56s, best_score 0.548
Local optimizer summary:
  Total iterations recorded: 4
  Avg iteration time: 257.15s
  Total LLM calls attributed to local iterations: 293
Per-iteration breakdown:
  Iter 1: time 134.66s, llm_calls 36
  Iter 2: time 433.66s, llm_calls 128
  Iter 1: time 107.62s, llm_calls 32
  Iter 2: time 352.64s, llm_calls 97


In [8]:
print("BEST PROMPT FOUND:")
print("=" * 80)
print(best_node.prompt_text)
print("=" * 80)
print(f"Score: {best_node.metrics.composite_score():.3f}")
print(f"Generation: {best_node.generation}")
print(f"Source: {best_node.source.value}")

BEST PROMPT FOUND:
Extract the precise answer span to respond to the question or output 'No answer' if the question cannot be answered based on the context.
Score: 0.548
Generation: 2
Source: global


In [9]:
print("BASELINE vs OPTIMIZED — Test Set")
print("=" * 80)

comparison = optimizer.compare_with_baseline(initial_prompt, test_examples)

print()
print("Summary:")
print(f"  {'Metric':<25s}  {'Baseline':>10s}  {'Optimized':>10s}  {'Delta':>10s}")
print(f"  {'-'*25}  {'-'*10}  {'-'*10}  {'-'*10}")
for name in ["exact_match", "token_f1"]:
   b = comparison["baseline"].get(name, 0.0)
   o = comparison["optimized"].get(name, 0.0)
   d = comparison["improvements"].get(name, 0.0)
   print(f"  {name:<25s}  {b:10.3f}  {o:10.3f}  {d:+10.3f}")
b_c = comparison["baseline"]["composite_score"]
o_c = comparison["optimized"]["composite_score"]
print(f"  {'COMPOSITE':<25s}  {b_c:10.3f}  {o_c:10.3f}  {o_c - b_c:+10.3f}")

BASELINE vs OPTIMIZED — Test Set

Comparing with baseline...
[diag] evaluate_prompt: execute=True sample=True stage=3 incoming=300 will_use=100 weights={exact_match:0.8, token_f1:0.2} llm_calls_at_start=329
[diag] execute_prompt_batch: total=100 batch_size=25 n_batches=4 llm_calls_before=329
[diag]   batch 1/4: OK (25/100 done, llm_calls=330)
[diag]   batch 2/4: OK (50/100 done, llm_calls=331)
[diag]   batch 3/4: OK (75/100 done, llm_calls=332)
[diag]   batch 4/4: OK (100/100 done, llm_calls=333)
[diag] execute_prompt_batch done: llm_calls_delta=4 total_calls=333
[diag] evaluate_prompt execution done: llm_calls_for_execution=4
[diag]   example[0] expected='No answer' actual='120 m (390 ft)'
[diag]   example[1] expected='productivity gap' actual='productivity gap between highly-paid professions and lower-paid professions'
[diag]   example[2] expected='No answer' actual='construction materials used'
[diag]   ... and 97 more examples
[diag]   metric 'exact_match': 0.2500 (stage=1 weight=0

In [10]:
print(optimizer.visualize_optimization_trajectory())


OPTIMIZATION TRAJECTORY

Generation | Best Score | Overall Best | Improvement
------------------------------------------------------------
   1       | 0.511      | 0.511       | +0.046 █████████████████████████
   2       | 0.548      | 0.548       | +0.036 ███████████████████████████




In [11]:
report = optimizer.get_optimization_report()

print("OPTIMIZATION REPORT:")
print("Overall Statistics:")
print(f"   Total time: {report['optimization_info']['total_time_seconds']:.2f}s")
print(f"   Generations: {report['optimization_info']['generations']}")
print(f"   Total nodes explored: {report['component_statistics']['history']['total_nodes']}")

print("Component Statistics:")
print(f"   Local optimizer iterations: {report['component_statistics']['local_optimizer']['total_iterations']}")
print(f"   Local improvements: {report['component_statistics']['local_optimizer']['improvements_count']}")
print(f"   Global optimizer steps: {report['component_statistics']['global_optimizer']['total_global_steps']}")
print(f"   Successful global changes: {report['component_statistics']['global_optimizer']['successful_global_changes']}")

print("Best Global Strategies:")
for i, strategy in enumerate(report['best_global_strategies'][:3], 1):
    print(f"   {i}. Score {strategy['score']:.3f}")
    print(f"      {strategy['strategy']['description'][:70]}...")


OPTIMIZATION REPORT:
Overall Statistics:
   Total time: 1210.62s
   Generations: 2
   Total nodes explored: 20
Component Statistics:
   Local optimizer iterations: 4
   Local improvements: 2
   Global optimizer steps: 1
   Successful global changes: 2
Best Global Strategies:
   1. Score 0.548
      meta-optimizer-single...
   2. Score 0.536
      meta-optimizer-single...
   3. Score 0.501
      meta-optimizer-single...


In [12]:
lineage = optimizer.history.get_lineage(best_node.id)

print("EVOLUTION OF BEST PROMPT:")
print("="*80)

for i, node in enumerate(lineage):
    print(f"Step {i}: Generation {node.generation}, Source: {node.source.value}")
    if node.is_evaluated:
        print(f"  Score: {node.metrics.composite_score():.3f}")

    if node.operations:
        print(f"  Operations:")
        for op in node.operations:
            print(f"    - {op.description}...")

    if i < len(lineage) - 1:  
        print("  ↓")


EVOLUTION OF BEST PROMPT:
Step 0: Generation 0, Source: initial
  Score: 0.465
  ↓
Step 1: Generation 1, Source: local
  Score: 0.511
  Operations:
    - MC synonym/paraphrase...
  ↓
Step 2: Generation 2, Source: local
  Score: 0.510
  Operations:
    - MC synonym/paraphrase...
  ↓
Step 3: Generation 3, Source: local
  Score: 0.512
  Operations:
    - MC synonym/paraphrase...
  ↓
Step 4: Generation 2, Source: global
  Score: 0.548
  Operations:
    - Meta-optimizer call 2 (gen 2)...


In [13]:
print("FINAL SUMMARY")
print("="*80)

history_stats = optimizer.history.get_statistics()
local_stats = optimizer.local_optimizer.get_statistics()
global_stats = optimizer.global_optimizer.get_statistics()

print("Overall Statistics:")
print(f"  Total nodes explored: {history_stats['total_nodes']}")
print(f"  Evaluations performed: {history_stats['evaluated_nodes']}")
print(f"  Generations completed: {history_stats['max_generation']}")
print(f"  Best score achieved: {history_stats['best_score']:.3f}")
print(f"  Average score: {history_stats['avg_score']:.3f}")

print("Local Optimization:")
print(f"  Total iterations: {local_stats['total_iterations']}")
print(f"  Improvements found: {local_stats['improvements_count']}")
print(f"  Success rate: {local_stats['improvement_rate']:.1%}")

print("Global Optimization:")
print(f"  Total global steps: {global_stats['total_global_steps']}")
print(f"  Candidates generated: {global_stats['total_candidates_generated']}")
print(f"  Successful changes: {global_stats['successful_global_changes']}")
print(f"  Success rate: {global_stats['success_rate']:.1%}")

print("Optimization complete!")
print(f"   Results saved to: ./optimization_results/")

FINAL SUMMARY
Overall Statistics:
  Total nodes explored: 20
  Evaluations performed: 20
  Generations completed: 3
  Best score achieved: 0.548
  Average score: 0.501
Local Optimization:
  Total iterations: 4
  Improvements found: 2
  Success rate: 50.0%
Global Optimization:
  Total global steps: 1
  Candidates generated: 6
  Successful changes: 2
  Success rate: 33.3%
Optimization complete!
   Results saved to: ./optimization_results/
